In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

In [2]:
file_path = 'C:/Users/mannu/2D Protien Folding/RS126.data.txt'
with open(file_path, 'r') as file:
    lines = file.readlines()

sequences, structures = [], []
for i in range(0, len(lines) - 1, 2):
    seq, struct = lines[i].strip(), lines[i+1].strip()
    if len(seq) == len(struct) and len(seq) > 0:
        sequences.append(seq)
        structures.append(struct)

# 2. Slice sliding windows
window_size = 13
pad_length = window_size // 2
X_data, Y_data = [], []
for seq, struct in zip(sequences[:50], structures[:50]):
    padded_seq = ("X" * pad_length) + seq + ("X" * pad_length)
    for j in range(len(seq)):
        X_data.append(padded_seq[j : j + window_size])
        Y_data.append(struct[j])


In [3]:
(X_data[0:10], Y_data[0:10])

(['XXXXXXAPAFSVS',
  'XXXXXAPAFSVSP',
  'XXXXAPAFSVSPA',
  'XXXAPAFSVSPAS',
  'XXAPAFSVSPASG',
  'XAPAFSVSPASGA',
  'APAFSVSPASGAS',
  'PAFSVSPASGASD',
  'AFSVSPASGASDG',
  'FSVSPASGASDGQ'],
 ['C', 'C', 'E', 'E', 'E', 'E', 'E', 'C', 'C', 'C'])

In [4]:
alphabet = "ACDEFGHIKLMNPQRSTVWYX"
char_to_idx = {char: i for i, char in enumerate(alphabet)}
vocab_size = len(alphabet) # 21

X_flat = torch.zeros(len(X_data), window_size * vocab_size)
for row_idx, window in enumerate(X_data):
    for char_idx, char in enumerate(window):
        if char in char_to_idx:
            col_idx = (char_idx * vocab_size) + char_to_idx[char]
            X_flat[row_idx, col_idx] = 1.0

# 4. Reshape to 3D Tensor for CNN: [Total_Rows, 21, 13]
X_cnn = X_flat.view(-1, window_size, vocab_size).transpose(1, 2)

# 5. Map target shapes to integers and create DataLoader
shape_mapping = {'C': 0, 'E': 1, 'H': 2}
Y_ints = [shape_mapping[shape] for shape in Y_data]
Y_tensor = torch.tensor(Y_ints, dtype=torch.long)

dataset = TensorDataset(X_cnn, Y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

print("Baseline Data ready!")
print(f"X_cnn shape: {X_cnn.shape}  [Batch, Channels, Length]")
print(f"Y_tensor shape: {Y_tensor.shape}")

Baseline Data ready!
X_cnn shape: torch.Size([8289, 21, 13])  [Batch, Channels, Length]
Y_tensor shape: torch.Size([8289])


In [5]:
class MiniFoldCNN(nn.Module):
    def __init__(self, output_size=3):
        super(MiniFoldCNN, self).__init__()
        
        # Conv Layer 1: In=21, Out=64
        self.conv1 = nn.Conv1d(in_channels=21, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
        # Conv Layer 2: In=64, Out=32
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=32, kernel_size=3, padding=1)
        
        # Final Linear Classification Layer: 32 channels * 13 length = 416 features
        self.fc = nn.Linear(32 * 13, output_size)
        
    def forward(self, x):
        # x entering shape: [Batch, 21, 13]
        x = self.conv1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x)
        x = self.relu(x)
        
        # Flatten [Batch, 32, 13] -> [Batch, 416]
        x = x.view(x.size(0), -1) 
        
        # Output Logits -> [Batch, 3]
        out = self.fc(x) 
        return out

# Initialize baseline model
model = MiniFoldCNN(output_size=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Original Baseline CNN Model initialized!")
print(model)

Original Baseline CNN Model initialized!
MiniFoldCNN(
  (conv1): Conv1d(21, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (conv2): Conv1d(64, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (fc): Linear(in_features=416, out_features=3, bias=True)
)


In [6]:
NUM_EPOCHS = 100

print("--- STARTING BASELINE CNN TRAINING ---")
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    
    for batch_X, batch_Y in train_loader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_Y)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Average Loss: {avg_loss:.4f}")

print("--- TRAINING COMPLETE ---")

--- STARTING BASELINE CNN TRAINING ---
Epoch [1/100] | Average Loss: 0.9982
Epoch [2/100] | Average Loss: 0.8429
Epoch [3/100] | Average Loss: 0.7991
Epoch [4/100] | Average Loss: 0.7707
Epoch [5/100] | Average Loss: 0.7334
Epoch [6/100] | Average Loss: 0.7038
Epoch [7/100] | Average Loss: 0.6784
Epoch [8/100] | Average Loss: 0.6619
Epoch [9/100] | Average Loss: 0.6456
Epoch [10/100] | Average Loss: 0.6280
Epoch [11/100] | Average Loss: 0.6213
Epoch [12/100] | Average Loss: 0.6085
Epoch [13/100] | Average Loss: 0.5909
Epoch [14/100] | Average Loss: 0.5840
Epoch [15/100] | Average Loss: 0.5823
Epoch [16/100] | Average Loss: 0.5711
Epoch [17/100] | Average Loss: 0.5593
Epoch [18/100] | Average Loss: 0.5526
Epoch [19/100] | Average Loss: 0.5515
Epoch [20/100] | Average Loss: 0.5390
Epoch [21/100] | Average Loss: 0.5247
Epoch [22/100] | Average Loss: 0.5295
Epoch [23/100] | Average Loss: 0.5197
Epoch [24/100] | Average Loss: 0.5145
Epoch [25/100] | Average Loss: 0.5045
Epoch [26/100] | Ave

In [7]:
def predict_protein_structure_baseline(protein_seq, trained_model, window_size=13, alphabet="ACDEFGHIKLMNPQRSTVWYX"):
    trained_model.eval()
    
    pad_length = window_size // 2
    padded_seq = ("X" * pad_length) + protein_seq + ("X" * pad_length)
    
    char_to_idx = {char: i for i, char in enumerate(alphabet)}
    vocab_size = len(alphabet)
    protein_len = len(protein_seq)
    
    # 1. Build flat matrix: [Protein_Length, 273]
    X_flat = torch.zeros(protein_len, window_size * vocab_size)
    for i in range(protein_len):
        window = padded_seq[i : i + window_size]
        for char_idx, char in enumerate(window):
            if char in char_to_idx:
                col_idx = (char_idx * vocab_size) + char_to_idx[char]
                X_flat[i, col_idx] = 1.0
                
    # 2. Reshape to 3D Tensor: [Protein_Length, 21, 13]
    X_inference = X_flat.view(-1, window_size, vocab_size).transpose(1, 2)
    
    # 3. Forward Pass
    with torch.no_grad():
        outputs = trained_model(X_inference)
        predicted_indices = torch.argmax(outputs, dim=1)
        
    # 4. Map integers back to letters
    int_to_shape = {0: 'C', 1: 'E', 2: 'H'}
    predicted_chars = [int_to_shape[int(idx.item())] for idx in predicted_indices]
    
    return "".join(predicted_chars)

# --- RUN INFERENCE TEST ---
test_input = "FVNQHLCGSHLVEALYLVCGERGFFYTPKA"
expected_output = "CCCCCCCCHHHHHHHHHHHHHHCECCCCCC"

baseline_pred = predict_protein_structure_baseline(test_input, model)

matches = sum(1 for p, e in zip(baseline_pred, expected_output) if p == e)
accuracy = (matches / len(expected_output)) * 100

print("--- BASELINE CNN INFERENCE TEST ---")
print(f"Input:    {test_input}")
print(f"Expected: {expected_output}")
print(f"Pred:     {baseline_pred}")
print(f"Accuracy: {accuracy:.1f}% ({matches}/{len(expected_output)} correct amino acids)")

--- BASELINE CNN INFERENCE TEST ---
Input:    FVNQHLCGSHLVEALYLVCGERGFFYTPKA
Expected: CCCCCCCCHHHHHHHHHHHHHHCECCCCCC
Pred:     CCEEEECHHHHHHHEEEEECCHCCHECHHC
Accuracy: 43.3% (13/30 correct amino acids)
